# Pluto M2 on Lightning AI: finish the evaluation

Finishes M2 from the runs and data saved on the private Hugging Face repo (`<you>/pluto-runs`).
Nothing is retrained. Results are uploaded to the Hub after every phase, and a re-run skips
phases that are already saved, so a stopped Studio loses nothing.

**Studio setup (once):**
- Choose a GPU machine (T4 or L4 is enough) for the Studio.
- Add environment variables / secrets: `HF_TOKEN` (required, Hugging Face write token) and
  `GITHUB_TOKEN` (only for the final results push). Restart the notebook kernel after adding them.
- Studio files persist between sessions, so setup is much faster the second time.

**Order (most important first):** 1) baseline evaluation, 2) aligned floor (30 chunks) + identity,
3) diagnostics at temperature 0.5 and 0, 4) demo audio, 5) reports, 6) push results.

Off-the-shelf components (disclosed): Kokoro-82M TTS (data synthesis), Kyutai Mimi codec, Whisper small.en.

## 1. Setup (re-run after every restart; fast once installed)

In [ ]:
import os, subprocess, sys, glob, importlib
REPO = "thedeepakreddy/Pluto-LLM"
BRANCH = "main"
HOME = os.path.expanduser("~")          # persistent Studio folder
WORK = os.path.join(HOME, "Pluto-LLM")
PY = os.path.join(HOME, "py312")
gh = os.environ.get("GITHUB_TOKEN")
url = f"https://{gh}@github.com/{REPO}.git" if gh else f"https://github.com/{REPO}.git"
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, WORK], check=True)
else:
    subprocess.run(["git", "-C", WORK, "fetch", "-q", url, BRANCH], check=False)
    subprocess.run(["git", "-C", WORK, "reset", "-q", "--hard", "FETCH_HEAD"], check=False)
%cd {WORK}
!(sudo apt-get install -y -q espeak-ng || apt-get install -y -q espeak-ng) > /dev/null 2>&1; which espeak-ng || echo "WARNING: espeak-ng missing"
# Isolated Python 3.12 env, as on Kaggle/Colab (Kokoro needs Python <= 3.12).
!pip install -q uv
if not os.path.exists(f"{PY}/bin/python"):
    !uv venv -q --python 3.12 {PY}
!uv pip install -q --python {PY}/bin/python -e ".[speech,hub]"
!uv pip install -q --python {PY}/bin/python "en_core_web_sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl"
LLM = f"{PY}/bin/llm"
!{LLM} --help > /dev/null && echo "llm CLI ready (Python 3.12)"
!pip install -q huggingface_hub
sys.path.insert(0, os.path.abspath("src"))
importlib.invalidate_caches()

HUB = ""
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import HfApi
    HUB = HfApi().whoami()["name"] + "/pluto-runs"
print("Hub:", HUB or "OFF: add the HF_TOKEN environment variable and restart the kernel")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
IDENTITY_DATASET = "thedeepakreddy/pluto-creator-identity"  # private HF dataset ("" = off)
ID = f"--set identity.hf_dataset={IDENTITY_DATASET} --set identity.speech_every=100" if IDENTITY_DATASET else ""
ALIGNED_RUN = "runs/m2-aligned-20261102-213616-72fb023"
BASELINE_RUN = "runs/m2-baseline-20261102-213616-72fb023"
WARM = "configs/m2-warmstart.yaml"
ALIGNED = "configs/m2-speech-aligned.yaml"

## 2. Data and runs from the Hub (skipped when already present)

In [ ]:
from huggingface_hub import snapshot_download
SPEECH_DIR = "speech/tinystories-speech-v1"
if not os.path.exists(f"{SPEECH_DIR}/manifest.json"):
    snapshot_download(HUB, allow_patterns=[SPEECH_DIR + "/*"], local_dir=".")
!ls {SPEECH_DIR}/manifest.json
# Tokenizer + identity probes (eval needs them; token shards are not needed for evaluation).
if not glob.glob("tokenizers/*/manifest.json"):
    !{LLM} prepare-data {WARM}
    !{LLM} train-tokenizer {WARM}
if IDENTITY_DATASET:
    !{LLM} prepare-identity {ALIGNED} {ID}
# Runs (checkpoints + any saved speech_eval*.json, so finished phases are reused).
for run in (ALIGNED_RUN, BASELINE_RUN):
    !{LLM} pull-run {HUB} {os.path.basename(run)}
!ls {ALIGNED_RUN}

## 3. Baseline evaluation (the main comparison) ~1 h

In [ ]:
!{LLM} eval-speech {BASELINE_RUN} 2>&1 | grep -v "Whisper did not predict"
!{LLM} speech-diagnose {BASELINE_RUN}/speech_eval.json

## 4. Aligned floor (30 chunks) + identity ~15-20 min (ceiling/model reused)

In [ ]:
!{LLM} eval-speech {ALIGNED_RUN} --floor --floor-n 30 2>&1 | grep -v "Whisper did not predict"
!{LLM} speech-diagnose {ALIGNED_RUN}/speech_eval.json

## 5. Diagnostics: less sampling randomness (50 chunks each) ~25-30 min

In [ ]:
for t in ("0.5", "0"):
    !{LLM} eval-speech {ALIGNED_RUN} --n 50 --temperature {t} 2>&1 | grep -v "Whisper did not predict"
    !{LLM} speech-diagnose {ALIGNED_RUN}/speech_eval_t{t}_k50.json

## 6. Demo audio

In [ ]:
from IPython.display import Audio, Markdown, display
from llm.speech.evaluate import DEMO_SENTENCES
for run in (ALIGNED_RUN, BASELINE_RUN):
    os.makedirs(f"{run}/demo", exist_ok=True)
    for i, text in enumerate(DEMO_SENTENCES[:3]):
        wav = f"{run}/demo/demo_{i}.wav"
        !{LLM} speak {run} --text "{text}" --out {wav} --seed 1
        print(run, "|", text)
        display(Audio(wav))

## 7. Reports

In [ ]:
for run in (ALIGNED_RUN, BASELINE_RUN):
    open(f"{run}/limitations.md", "w").write(
        "# Known limitations\n\n"
        "- ~30M parameters; speech is synthetic (one Kokoro voice) reading TinyStories only.\n"
        "- Audio quality is bounded by the Mimi codec; WER is measured by Whisper, itself imperfect "
        "(it hallucinates words from garbled audio).\n"
        "- The model reads the first sentence(s) of a passage well, then often drifts into repetition "
        "and does not stop (generated audio ~2.3x the reference length).\n"
        "- Single run per layout, single seed; the floor is scored on 30 chunks.\n"
        "- Trained on Kaggle (2x T4); evaluated on Colab and Lightning AI via checkpoint resume.\n")
    !{LLM} report {run}
!{LLM} compare {ALIGNED_RUN} {BASELINE_RUN}
display(Markdown(open(f"{ALIGNED_RUN}/report.md").read().replace("](charts/", f"]({ALIGNED_RUN}/charts/")))

## 8. Push results to GitHub (public artifacts only; identity redacted by `export_public_results`)

In [ ]:
def run_git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr + r.stdout).replace(gh, "***") if gh else r.stderr + r.stdout
        raise RuntimeError(f"git {args[0]} failed: {msg}")
    return r.stdout

if not gh:
    print("GITHUB_TOKEN not set; skipping push")
else:
    from llm.publish import export_public_results   # redacts creator identity; raises on any leak
    for run in (ALIGNED_RUN, BASELINE_RUN):
        export_public_results(run, f"results/m2/{os.path.basename(run)}")
    run_git("config", "user.name", "Pluto Lightning")
    run_git("config", "user.email", "lightning@users.noreply.github.com")
    run_git("add", "results")
    if not run_git("status", "--porcelain", "results").strip():
        print("nothing changed; not committing")
    else:
        run_git("commit", "-m", "results(m2): evaluation reports")
        run_git("pull", "--rebase", url, BRANCH)
        run_git("push", url, f"HEAD:{BRANCH}")
        print("pushed results/")